## Using the BJT models from glayout

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from glayout.primitives.bjt import draw_bjt
from glayout import MappedPDK, sky130 , gf180

In [ ]:
pdk=gf180
pdk.activate()

In [ ]:
bjt=draw_bjt(pdk,(5,5),"pnp")

In [ ]:
bjt.show()

In [ ]:
bjt.ports

In [ ]:
from glayout.primitives.bjt import multiplier, __mult_array_macro

In [ ]:
bjt=__mult_array_macro(pdk=pdk,active_area=(5,5),bjt_type="pnp",bc_rmult=2,multipliers=5)

In [ ]:
bjt.show()

In [ ]:
bjt.center

In [ ]:
bjt.pprint_ports()

In [ ]:
from glayout.primitives.bjt import pnp, npn

In [ ]:
bjt_pnp=pnp(pdk=pdk,active_area=(5,5),bc_rmult=2,multipliers=5)

In [ ]:
bjt_pnp.show()

In [ ]:
bjt_npn=npn(pdk=pdk,active_area=(5,5),bc_rmult=2,multipliers=5, with_dummy=(False,False))

In [ ]:
bjt_npn.show()

In [ ]:
from pathlib import Path
import os
import subprocess
import tempfile
magicrc_file = Path(os.environ['PDKPATH']) / "libs.tech" / "magic" / f"{os.environ['PDK']}.magicrc"
magicrc_file

In [ ]:
def extract_pex(design, path_to_dir):
    design_name=design.name
    if not path_to_dir.exists():
        path_to_dir.mkdir(parents=True, exist_ok=False)
    
    pex_path = path_to_dir / f"{design_name}_pex.spice"
    gds_path = path_to_dir / f"{design_name}.gds"
    
    design.write_gds(str(gds_path))
        
    magic_script_content = f"""
    drc off            
    gds flatglob *\\$\\$*
    gds read {gds_path}
    
    flatten {design_name}
    load {design_name}
    select top cell
    extract do local
    extract all
    ext2sim labels on
    ext2sim
    extresist tolerance 10
    extresist
    ext2spice lvs
    ext2spice cthresh 0
    ext2spice extresist on
    ext2spice -o {str(pex_path)}
    exit
    """
    
    with tempfile.NamedTemporaryFile(mode='w', delete=False) as magic_script_file:
        magic_script_file.write(magic_script_content)
        magic_script_path = magic_script_file.name
        
    magic_cmd = f"bash -c 'magic -rcfile {magicrc_file} -noconsole -dnull < {magic_script_path}'",
    magic_subproc = subprocess.run(
        magic_cmd, 
        shell=True,
        check=True,
        capture_output=True
    )
    
    magic_subproc_code = magic_subproc.returncode
    magic_subproc_out = magic_subproc.stdout.decode('utf-8')
    print(magic_subproc_out)

In [ ]:
path_to_dir = Path("/foss/designs/gLayout/tutorial").resolve() / "ext" / bjt_pnp.name
extract_pex(bjt_pnp, path_to_dir)

In [ ]:
path_to_dir = Path("/foss/designs/gLayout/tutorial").resolve() / "ext" / bjt_npn.name
extract_pex(bjt_npn, path_to_dir)